# Lekcija 18: Zavarovanje AI agentov s kriptografskimi prejemki

## Praktični zvezek

Ta zvezek vas vodi skozi štiri vaje:

1. **Podpišite svoj prvi prejem** za klic orodja agenta in ga preverite.
2. **Spremenite prejem** in opazujte, kako preverjanje ne uspe.
3. **Zgradite verigo s tremi prejemki** in potrdite integriteto verige.
4. **Ovijte klic orodja Microsoft Agent Framework**, tako da vsak ukrep oddaja prejem.

Vse kriptografske primitivne funkcije so uvožene iz dobro vzdrževanih knjižnic (`pynacl` za Ed25519, `jcs` za RFC 8785 kanonični JSON, `hashlib` iz standardne Pythonove knjižnice za SHA-256). Sam logični prejem je čist Python, ki ga lahko preberete in spremenite.

Zaženite celice po vrsti. Vsak odsek je kratek in samostojen.


## Namestitev

Namestite obe odvisnosti. Obe imata dovoljujoči licenci (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Pomočna orodja

Ta dva pripomočka obdelujeta kodiranje base64url (brez polnjenja) in hashiranje SHA-256 poljubnih objektov. Omogočata, da je preostali del zapisnika osredotočen na logiko prejemka samega.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Poglavje 1: Podpišite svoj prvi prejem

Predstavljajte si, da je naš agent za **Contoso Travel** pravkar poiskal lete iz Sydneya v Los Angeles za stranko. Želimo to klic orodja zabeležiti kot podpisan prejem, tako da ga bo lahko prihodnji revizor preveril brez zaupanja v nas.

### Korak 1.1: Ustvarite ključ za podpisovanje

V produkciji bi ključ za podpisovanje agenta živel v strojno varnostnem modulu (HSM), Azure Key Vault ali v podobnem zaščitenem shrambi. Za to lekcijo ustvarimo nov ključ v pomnilniku.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Korak 1.2: Ustvarite vsebino potrdila

Vsebuje vse, kar želimo, da potrdilo potrdi: kdo je ukrepal, kateri pripomoček, s kakšnimi argumenti, kaj se je vrnilo, pod katero politiko in kdaj. Argumente in rezultat zgoščujemo (hashamo), namesto da bi jih vključili neposredno, da potrdilo ne bi razkrilo občutljivih vsebin.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Korak 1.3: Podpišite in sestavite potrdilo

Trije koraki:

1. Normalizirajte vsebino z uporabo JCS, tako da dve implementaciji, ki proizvedeta isto logično potrdilo, ustvarita enake bajtne nizove.
2. Zgoščite normalizirane bajte s SHA-256.
3. Podpišite zgoščeno vrednost s privatnim ključem Ed25519.

Podpis se nato pritrdi na izvirno vsebino, da nastane končno potrdilo.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Korak 1.4: Preverite potrdilo

Preverjanje je obratni postopek. Odstranimo podpis, ponovno izračunamo kanonični zgošček in preverimo podpis glede na javni ključ v potrdilu.

Revizor, ki izvaja to preverjanje, od nas ne potrebuje ničesar razen samega potrdila. Ni treba klicati nobene storitve, poizvedovati v imeniku ključev ali vzpostavljati zaupanja.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Morali bi videti `Potrdilo je veljavno: True`. Agent je ustvaril svoj prvi kriptografsko podpisan auditni zapis.


## Poglavje 2: Spremeni račun

Glavni namen računov je, da so vidne morebitne spremembe. Dokažimo to.

Spremenili bomo natanko en znak na računu in opazovali, kako preverjanje ne uspe.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Kaj se je pravkar zgodilo?

Ko smo spremenili `policy_id`, so se spremenili kanonični bajti. SHA-256 hash teh bajtov se je spremenil. Podpis (ki je bil nad originalnim hashom) se ne ujema več z novim hashem. Preverjanje pravilno vrne `False`.

Ni mogoče spremeniti nobenega polja potrdila in ga še vedno potrditi, razen če napadalec nima zasebnega ključa. Dokler je zasebni ključ v ključni shrambi in je javni ključ objavljen, je manipulacije nemogoče prikriti.

Poskusi sam: spremeni `tool_name` ali `agent_id` ali `timestamp` v zgornji celici in ponovno zaženi. Vsaka sprememba ustvari neveljavno potrdilo.


## Poglavje 3: Povežite potrdila skupaj

En sam račun varuje eno dejanje. Večina agentov izvede veliko dejanj. Da bi celotno zaporedje naredili zaznavno za poseganje, povezujemo vsako potrdilo s prejšnjim tako, da vključimo heš prejšnjega potrdila v vsebino novega potrdila.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Če kdo odstrani ali prerazporedi potrdilo, se veriga na točno tej točki prekine. Preverjanje katerega koli kasnejšega potrdila ne uspe, ker `previous_receipt_hash` več ne ustreza dejanskemu hešu njegovega predhodnika.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Zdaj prekini verigo tako, da posežeš v srednji račun in ponovno preveriš. Spremenjeni račun ne uspe pri preverjanju podpisa, IN naslednji račun ne uspe pri preverjanju verige (ker njegov `previous_receipt_hash` ne ustreza več spremenjenemu hash vrednosti srednjega računa).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Prejemnica 0 se še vedno preveri (ni bila spremenjena in nima predhodnika, od katerega bi bila odvisna). Prejemnica 1 ne uspe pri preverjanju podpisa, ker smo spremenili `tool_args_hash`. Prejemnica 2 ne uspe pri preverjanju povezave verige, ker je bil njen `previous_receipt_hash` izračunan glede na prvotno (zdaj spremenjeno) prejemnico 1.

Tudi če napadalec ponovno podpiše spremenjeno prejemnico 1 (česar ne more storiti brez zasebnega ključa), bi neskladje povezanosti verige v prejemnici 2 še vedno razkrilo poseg. Da bi prikril spremembo, bi moral napadalec ponovno podpisati vse prejemnice od točke spremembe naprej, kar zahteva posedovanje zasebnega ključa.


## Oddelek 4: Ovijte klic orodja agenta s podpisom prejemka

V pravem okolju ne želite, da si vsak avtor agenta zapomni, da mora poklicati `make_receipt`. Želite, da je podpis prejemka samodejen pri vsakem klicu orodja.

Tukaj je najpreprostejši vzorec: ovojna razred, ki sprejme katerokoli klicateljno funkcijo orodja in vrne verzijo, ki izdaja prejemke. To se prilagodi kateremukoli agentnemu ogrodju, vključno z Microsoft Agent Framework (`agent_framework.foundry`).

Če nimate nastavljenega Microsoft Foundry projekta, lokalni poskus spodaj vseeno prikazuje ta vzorec.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


### Integracija z Microsoft Agent Framework

Zavitek `ReceiptedTool` zgoraj je neodvisen od ogrodja. Za uporabo znotraj agenta, zgrajenega z Microsoft Agent Framework, registrirajte zavito funkcijo kot orodje. Osnutek (namesto ponaredka bi uporabili dejansko registracijo orodja Microsoft Foundry):

```python
# Psevdokoda, ki prikazuje obliko integracije.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="Vi ste agent Contoso Travel ...",
#     tools=[receipted_lookup],   # ovito orodje, ne surova funkcija
# )
# response = agent.run("Poišči lete iz Sydneya v Los Angeles junija.")
#
# # Po izvedbi ima vsak klic orodja, ki ga je agent izvedel, podpisan prejemek:
# audit_chain = receipted_lookup.receipts
```

Agentno ogrodje ne potrebuje vedeti ničesar o prejemkih. Podpisovanje prejemkov je zavito okrog orodja, ni pritrjeno znotraj ogrodja. Tako dodate izvor obstoječi kodi agenta brez prepisovanja agenta.


## Povzetek in izziv za raztezanje

Imate:

- Generirali par ključev Ed25519.
- Sestavili in podpisali potrdilo za klic orodja agenta.
- Potrdilo preverili brez povezave z uporabo samo javnega ključa.
- Ponarejali potrdilo in opazovali neuspeh preverjanja.
- Sestavili zaporedje treh potrdil z verigo zgoščenk.
- Ponarejali sredino verige in opazovali tako neuspeh podpisa kot tudi neuspeh verižne povezave.
- Ovili funkcijo orodja z avtomatskim podpiranjem potrdil.

**Izziv za raztezanje.** Razširite shemo potrdila z `request_id` poljem (UUID za distribuirano sledenje). Posodobite `make_receipt`, da ga vključuje, in potrdite, da se potrdila še vedno preverjajo od začetka do konca. Nato spremenite polje po podpisu in potrdite, da preverjanje ne uspe. To vas prisili, da si notranje predstavljate, kako vsak bajt kanoničnega kodiranja prispeva k podpisu.

**Pomembna meja.** Potrdila dokazujejo tri stvari in le tri stvari: pripadnost (ta ključ je podpisal to vsebino), celovitost (vsebina se od podpisa ni spremenila) in vrstni red (to potrdilo je prišlo po tem potrdilu). Ne dokazujejo, da je bila dejanje agenta pravilno, da je bila politika z imenom `policy_id` dejansko ocenjena ali da je agent sledil vsakemu pravilu. Potrdila so temelj. Upravljanje je sistem, ki ga zgradite na tem temelju.

Še enkrat preberite README lekcije s to mejo v mislih. Najpogostejša napaka ekip s potrdili je domneva, da "imamo potrdila" pomeni "upravljajo nas." Ne pomeni. Potrdila omogočajo revidiranje vedenja agenta. Ne zagotavljajo, da je to vedenje pravilno.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
